# 5장 pandas 데이터 정리

이 노트북은 웹 교재 [5장 pandas 데이터 정리](https://pidocs.aifusionlab.org/ai-python/ch05/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 실습 자료 내려받기

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/seoul_weather_2021_2025.csv
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/class_rssi_20261104.csv
!ls *.csv

## 5.1 DataFrame 기초

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch05/5-1/)

In [ ]:
import pandas as pd

df = pd.read_csv("seoul_weather_2021_2025.csv", parse_dates=["date"])
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(1)

In [ ]:
hot = df[df["temp_max"] >= 33]
print(len(hot))
hot["date"].dt.year.value_counts().sort_index()

In [ ]:
df.sort_values("temp_max", ascending=False).head(5)[["date", "temp_max", "temp_mean"]]

## 5.2 결측치·이상치 처리

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch05/5-2/)

In [ ]:
rssi = pd.read_csv("class_rssi_20261104.csv")
print(rssi.shape)
print(rssi.isna().sum())
rssi.groupby("zone")["rssi"].agg(["count", "mean", "median"]).round(1)

In [ ]:
pv = rssi.pivot_table(index="node", columns="zone", values="rssi", aggfunc="median")
pv

In [ ]:
print(pv.isna().sum())
print("4개 지점을 모두 측정한 노드:", len(pv.dropna()))

In [ ]:
filled = pv.fillna(pv.median())
filled.loc[["N7", "S09"]]

In [ ]:
out_range = rssi[(rssi["rssi"] > 0) | (rssi["rssi"] < -100)]
print("범위 밖 기록:", len(out_range))

q1 = rssi.groupby("zone")["rssi"].quantile(0.25)
q3 = rssi.groupby("zone")["rssi"].quantile(0.75)
iqr = q3 - q1
low = rssi["zone"].map(q1 - 1.5 * iqr)
high = rssi["zone"].map(q3 + 1.5 * iqr)
outliers = rssi[(rssi["rssi"] < low) | (rssi["rssi"] > high)]
print("IQR 기준 이상치 후보:", len(outliers))
outliers.groupby("zone")["rssi"].agg(["count", "min", "max"])

In [ ]:
rssi["outlier"] = (rssi["rssi"] < low) | (rssi["rssi"] > high)
rssi["outlier"].value_counts()

## 5.3 날짜 열과 월별 집계

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch05/5-3/)

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df[["date", "year", "month", "temp_mean"]].head(3)

In [ ]:
monthly = df.groupby("month").agg(temp=("temp_mean", "mean"), precip=("precip_mm", "sum"))
monthly["precip"] = monthly["precip"] / 5      # 5년 합계를 연평균으로 환산
monthly.round(1)

In [ ]:
df["rain"] = df["precip_mm"] >= 1.0
(df.groupby("month")["rain"].sum() / 5).round(1)

In [ ]:
df.pivot_table(index="year", columns="month", values="temp_mean", aggfunc="mean").round(1)